# Zilliz vector database 

In [15]:
import json
import os
from pathlib import Path
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv

In [16]:
BASE_DIR = Path(r"D:\Final_GRAG")
load_dotenv(BASE_DIR / ".env")

ZILLIZ_CLOUD_URI = os.getenv("ZILLIZ_CLOUD_URI")
ZILLIZ_CLOUD_API_KEY = os.getenv("ZILLIZ_CLOUD_API_KEY")

UNITS_JSON_PATH = BASE_DIR / "metadata" / "gri_units" / "gri_units.json"
EDGES_JSON_PATH = BASE_DIR / "metadata" / "gri_edges" / "gri_edges.json"

In [17]:
# Tạo kết nối tới Zilliz Cloud bằng pymilvus
client = MilvusClient(
    uri=ZILLIZ_CLOUD_URI,
    token=ZILLIZ_CLOUD_API_KEY
)

## Collection 1: gri_units

In [18]:
gri_units_schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)

gri_units_schema.add_field(
    field_name="unit_id",
    datatype=DataType.VARCHAR,
    max_length=64,
    is_primary=True
)

gri_units_schema.add_field(
    field_name="dense_embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=1024
)

gri_units_schema.add_field(
    field_name="sparse_embedding",
    datatype=DataType.SPARSE_FLOAT_VECTOR
)

gri_units_schema.add_field(
    field_name="standard_id",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="standard_name",
    datatype=DataType.VARCHAR,
    max_length=256
)

gri_units_schema.add_field(
    field_name="standard_type",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="disclosure_id",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="disclosure_name",
    datatype=DataType.VARCHAR,
    max_length=512
)

gri_units_schema.add_field(
    field_name="requirement_id",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="requirement_text",
    datatype=DataType.VARCHAR,
    max_length=8192
)

gri_units_schema.add_field(
    field_name="requirement_type",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="claim_level",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="is_mandatory",
    datatype=DataType.BOOL
)

gri_units_schema.add_field(
    field_name="year",
    datatype=DataType.INT16
)

gri_units_schema.add_field(
    field_name="sector_applicability",
    datatype=DataType.VARCHAR,
    max_length=512
)

gri_units_schema.add_field(
    field_name="hierarchy_level",
    datatype=DataType.INT8
)

gri_units_schema.add_field(
    field_name="parent_requirement",
    datatype=DataType.VARCHAR,
    max_length=32
)

{'auto_id': False, 'description': '', 'fields': [{'name': 'unit_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 64}, 'is_primary': True, 'auto_id': False}, {'name': 'dense_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}, {'name': 'sparse_embedding', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>}, {'name': 'standard_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 32}}, {'name': 'standard_name', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 256}}, {'name': 'standard_type', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 32}}, {'name': 'disclosure_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 32}}, {'name': 'disclosure_name', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 512}}, {'name': 'requirement_id', 'description': '', 'type': <DataType.

In [19]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="unit_id"
)

# Index cho Dense vector (semantic/cosine similarity search)
index_params.add_index(
    field_name="dense_embedding",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)

# Index cho Sparse vector
index_params.add_index(
    field_name="sparse_embedding",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP"
)

index_params.add_index(field_name="standard_id")
index_params.add_index(field_name="standard_type")
index_params.add_index(field_name="disclosure_id")
index_params.add_index(field_name="claim_level")
index_params.add_index(field_name="is_mandatory")

In [20]:
if "gri_units" in client.list_collections():
    client.drop_collection("gri_units")

# Tạo collection gri_units mới
client.create_collection(
    collection_name="gri_units",
    schema=gri_units_schema,
    index_params=index_params
)

In [21]:
with open(UNITS_JSON_PATH, 'r', encoding='utf-8') as f:
    units_data = json.load(f)

print(len(units_data))

618


In [22]:
prepared_units = []
for unit in units_data:
    prepared_unit = unit.copy()

    if 'sparse_embedding' in prepared_unit:
        sparse = prepared_unit['sparse_embedding']
        if isinstance(sparse, dict):
            if 'indices' in sparse and 'values' in sparse:
                indices = sparse['indices']
                values = sparse['values']
                prepared_unit['sparse_embedding'] = dict(zip(indices, values))
        else:
            print("Sai định dạng sparse embedding")

    if prepared_unit.get('parent_requirement') is None:
        prepared_unit['parent_requirement'] = ""
    
    prepared_units.append(prepared_unit)

print(len(prepared_units))

618


In [23]:
batch_size = 100

for i in range(0, len(prepared_units), batch_size):
    batch = prepared_units[i:i+batch_size]
    result = client.insert(
        collection_name="gri_units",
        data=batch
    )

## Collection 2: gri_edges

In [24]:
gri_edges_schema = MilvusClient.create_schema(
    auto_id=False,  
    enable_dynamic_field=True,
)

gri_edges_schema.add_field(
    field_name="edge_id",
    datatype=DataType.INT64,
    is_primary=True
)

gri_edges_schema.add_field(
    field_name="source_unit_id",
    datatype=DataType.VARCHAR,
    max_length=64
)

gri_edges_schema.add_field(
    field_name="target_unit_id",
    datatype=DataType.VARCHAR,
    max_length=64
)

gri_edges_schema.add_field(
    field_name="edge_type",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_edges_schema.add_field(
    field_name="edge_weight",
    datatype=DataType.FLOAT
)

gri_edges_schema.add_field(
    field_name="description",
    datatype=DataType.VARCHAR,
    max_length=512
)

gri_edges_schema.add_field(
    field_name="is_required_for_compliance",
    datatype=DataType.BOOL
)

gri_edges_schema.add_field(
    field_name="dummy_vector",
    datatype=DataType.FLOAT_VECTOR,
    dim=2
)

{'auto_id': False, 'description': '', 'fields': [{'name': 'edge_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'source_unit_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 64}}, {'name': 'target_unit_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 64}}, {'name': 'edge_type', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 32}}, {'name': 'edge_weight', 'description': '', 'type': <DataType.FLOAT: 10>}, {'name': 'description', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 512}}, {'name': 'is_required_for_compliance', 'description': '', 'type': <DataType.BOOL: 1>}, {'name': 'dummy_vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 2}}], 'enable_dynamic_field': True, 'enable_namespace': False}

In [25]:
edges_index_params = client.prepare_index_params()

edges_index_params.add_index(field_name="edge_id")

# Index cho Dummy vector 
edges_index_params.add_index(
    field_name="dummy_vector",
    index_type="AUTOINDEX",
    metric_type="L2"
)

edges_index_params.add_index(field_name="source_unit_id")
edges_index_params.add_index(field_name="target_unit_id")
edges_index_params.add_index(field_name="edge_type")
edges_index_params.add_index(field_name="is_required_for_compliance")

In [26]:
if "gri_edges" in client.list_collections():
    client.drop_collection("gri_edges")
    
# Tạo collection gri_edges mới
client.create_collection(
    collection_name="gri_edges",
    schema=gri_edges_schema,
    index_params=edges_index_params
)

In [27]:
with open(EDGES_JSON_PATH, 'r', encoding='utf-8') as f:
    edges_data = json.load(f)

# Thêm vector dummy vào mỗi cạnh
for edge in edges_data:
    edge['dummy_vector'] = [0.0, 0.0]  

print(len(edges_data))

756


In [28]:
batch_size = 200

for i in range(0, len(edges_data), batch_size):
    batch = edges_data[i:i+batch_size]
    result = client.insert(
        collection_name="gri_edges",
        data=batch
    )